In [ ]:
!pip install -q numpy pandas scikit-learn faiss-cpu gradio matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 41.6 MB/s eta 0:00:00


In [ ]:
import re
from pathlib import Path

import faiss
import gradio as gr
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize


In [ ]:
df = pd.read_csv("/content/Jadarat_data.csv")
df = df.loc[:, ~df.columns.str.contains("Unnamed")]

arabic_diacritics = re.compile(r"ّ|َ|ً|ُ|ٌ|ِ|ٍ|ْ|ـ")

def normalize_arabic(text: str) -> str:
    text = str(text)
    text = re.sub(arabic_diacritics, "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["job_text"] = (
    df.get("المسمى الوظيفي", "").astype(str) + " " +
    df.get("الوصف الوظيفي", "").astype(str) + " " +
    (df.get("المدينة", "").astype(str) + " ") * 3 +
    df.get("المسمى الوظيفي", "").astype(str)
)
df["job_text"] = df["job_text"].apply(normalize_arabic)

print("Dataset size:", df.shape)


Dataset size: (7041, 8)


In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
job_vectors = vectorizer.fit_transform(df["job_text"])
job_vectors_dense = normalize(job_vectors.toarray().astype(np.float32), norm="l2", axis=1)

faiss_index = faiss.IndexFlatIP(job_vectors_dense.shape[1])
faiss_index.add(job_vectors_dense)

def retrieve_job(user_query: str, k: int = 1) -> pd.Series:
    cleaned = normalize_arabic(user_query)
    q_vec = vectorizer.transform([cleaned]).toarray().astype(np.float32)
    q_vec = normalize(q_vec, norm="l2", axis=1)
    _, I = faiss_index.search(q_vec, k)
    return df.iloc[I[0][0]]


In [ ]:
def build_recommendation(row: pd.Series) -> str:
    title = row.get("المسمى الوظيفي", "وظيفة")
    city = row.get("المدينة", "مدينة غير مذكورة")
    desc = str(row.get("الوصف الوظيفي", ""))[:220]
    return (
        f"نرشح لك وظيفة {title} في {city}. راجع المتطلبات التالية:\n"
        f"{desc}\n"
        "ننصح بالتأكد من توافق خبرتك مع المهام المذكورة والتقديم عبر المنصة."
    )


In [ ]:
def job_chatbot(message, history):
    row = retrieve_job(message)
    response = f"""
وجدنا وظيفة مناسبة لك:

المسمى الوظيفي: {row.get('المسمى الوظيفي', 'غير متوفر')}
الشركة: {row.get('الجهة', 'غير مذكور')}
المدينة: {row.get('المدينة', 'غير مذكور')}

الوصف الوظيفي:
{str(row.get('الوصف الوظيفي', 'غير متوفر'))[:500]}...

توصية إضافية:
{build_recommendation(row)}
"""
    return response


In [ ]:
demo = gr.ChatInterface(
    fn=job_chatbot,
    title="كفاءات",
    description="أدخل مؤهلك وخبرتك ومدينتك وسيتم ترشيح وظائف مناسبة وتوليد توصية نصية عربية",
)

demo.launch(share=True)


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ba612e4aa75c97b13a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
